# Gold Review Fact

This notebook builds the `fact_reviews` Gold model from the cleaned Silver order reviews dataset.

**Grain:** One row per `review_id` and `order_id`.

In [0]:
from pyspark.sql import functions as F

## 1. Define storage paths

In [0]:
SILVER_ORDER_REVIEWS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/order_reviews"
)

GOLD_FACT_REVIEWS_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/fact_reviews"
)

print(f"Silver source: {SILVER_ORDER_REVIEWS_PATH}")
print(f"Gold target: {GOLD_FACT_REVIEWS_PATH}")

## 2. Read Silver order reviews

In [0]:
silver_order_reviews_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDER_REVIEWS_PATH)
)

silver_review_count = silver_order_reviews_df.count()

print(f"Silver review rows: {silver_review_count:,}")

display(silver_order_reviews_df.limit(10))

## 3. Validate required columns

In [0]:
required_columns = {
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
    "_silver_processed_at",
}

missing_columns = required_columns - set(silver_order_reviews_df.columns)

if missing_columns:
    raise ValueError(
        "Silver order reviews is missing required columns: "
        f"{sorted(missing_columns)}"
    )

print("Required column validation passed.")

## 4. Build review fact

In [0]:
fact_reviews_df = (
    silver_order_reviews_df
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "_silver_processed_at",
    )
    .withColumn(
        "review_creation_date_key",
        F.date_format(
            F.to_date("review_creation_date"),
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn(
        "review_answer_date_key",
        F.date_format(
            F.to_date("review_answer_timestamp"),
            "yyyyMMdd",
        ).cast("int"),
    )
    .withColumn(
        "has_review_comment_title",
        F.col("review_comment_title").isNotNull()
        & (F.trim(F.col("review_comment_title")) != ""),
    )
    .withColumn(
        "has_review_comment_message",
        F.col("review_comment_message").isNotNull()
        & (F.trim(F.col("review_comment_message")) != ""),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(fact_reviews_df.limit(10))

## 5. Validate review fact

In [0]:
fact_review_count = fact_reviews_df.count()

duplicate_review_count = (
    fact_reviews_df
    .groupBy("review_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_grain_count = (
    fact_reviews_df
    .filter(
        F.col("review_id").isNull()
        | F.col("order_id").isNull()
    )
    .count()
)

invalid_review_score_count = (
    fact_reviews_df
    .filter(
        F.col("review_score").isNull()
        | ~F.col("review_score").between(1, 5)
    )
    .count()
)

if fact_review_count == 0:
    raise ValueError("Review fact is empty.")

if fact_review_count != silver_review_count:
    raise ValueError(
        "Review fact row count does not match Silver reviews. "
        f"Silver: {silver_review_count:,}, "
        f"Gold: {fact_review_count:,}"
    )

if duplicate_review_count > 0:
    raise ValueError(
        "Review fact contains "
        f"{duplicate_review_count:,} duplicate grain combinations."
    )

if null_grain_count > 0:
    raise ValueError(
        f"Review fact contains {null_grain_count:,} rows with null grain keys."
    )

if invalid_review_score_count > 0:
    raise ValueError(
        "Review fact contains "
        f"{invalid_review_score_count:,} invalid review scores."
    )

print(f"Review fact rows: {fact_review_count:,}")
print("Review fact grain validation passed.")

## 6. Write review fact to Gold

In [0]:
(
    fact_reviews_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_FACT_REVIEWS_PATH)
)

print(f"Review fact written to: {GOLD_FACT_REVIEWS_PATH}")

## 7. Validate Gold output

In [0]:
written_fact_reviews_df = (
    spark.read
    .format("delta")
    .load(GOLD_FACT_REVIEWS_PATH)
)

written_review_count = written_fact_reviews_df.count()

if written_review_count != fact_review_count:
    raise ValueError(
        "Gold review fact write validation failed. "
        f"Expected: {fact_review_count:,}, "
        f"Written: {written_review_count:,}"
    )

print(f"Written review fact rows: {written_review_count:,}")
print("Gold review fact write validation passed.")

## 8. Inspect Gold review fact

In [0]:
written_fact_reviews_df.printSchema()

display(
    written_fact_reviews_df
    .orderBy("review_id", "order_id")
    .limit(10)
)